In [ ]:
!pip install xgboost optuna --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap
import warnings, os
from pathlib import Path
import yfinance as yf

import xgboost as xgb
import optuna
from optuna.samplers import TPESampler

In [ ]:
np.random.seed(20160101)
os.makedirs("results", exist_ok=True)

### Data Collection

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 1.  UNIVERSE & PRICE DATA
# ══════════════════════════════════════════════════════════════════════════════

# Populated after fetch_prices() runs in main()
TICKERS: list[str] = []

_BASELINE_APR2016 = {
    "AAPL", "AXP", "BA",  "CAT",  "CSCO", "CVX",  "DD",
    "DIS",  "GE",  "GS",  "HD",   "IBM",  "INTC", "JNJ",
    "JPM",  "KO",  "MCD", "MMM",  "MRK",  "MSFT", "NKE",
    "PFE",  "PG",  "RTX", "TRV",  "UNH",  "V",    "VZ",
    "WMT",  "XOM",
}

_CHANGES = [
    ("2018-06-26", ["WBA"],              ["GE"]),
    ("2019-04-02", ["DOW"],              ["DD"]),
    ("2020-04-06", ["RTX"],              ["UTX"]),
    ("2020-08-31", ["AMGN","CRM","HON"], ["XOM","PFE","RTX"]),
    ("2024-02-26", ["AMZN","SHW"],       ["WBA","INTC"]),
    ("2024-11-01", ["NVDA"],             ["DOW"]),
]

def fetch_prices(
    start_date: str = "2016-04-01",
    end_date:   str = "2026-04-18",
) -> pd.DataFrame:
    """
    Download daily adjusted close prices for all historical Dow Jones 30
    constituents via yfinance, with point-in-time constituent tracking.
    Returns a DataFrame of adjusted close prices (forward-filled up to 10 days).
    """
    # Build full historical ticker universe
    all_tickers = set(_BASELINE_APR2016)
    for _, added, removed in _CHANGES:
        all_tickers.update(added)
    # WBA has data issues; UTX was renamed to RTX before the study period
    all_tickers -= {"WBA", "UTX"}
    all_tickers = sorted(all_tickers)

    raw    = yf.download(all_tickers, start=start_date, end=end_date,
                         auto_adjust=True, progress=False)
    prices = raw["Close"].ffill(limit=10)
    prices = prices.dropna(axis=1, how="all")
    return prices


def get_constituents_on_date(date: pd.Timestamp) -> set:
    """Return the exact Dow 30 constituents on a given date."""
    constituents = set(_BASELINE_APR2016)
    for change_date_str, added, removed in _CHANGES:
        if date >= pd.Timestamp(change_date_str):
            constituents.update(added)
            constituents -= set(removed)
    # WBA/UTX excluded as before
    constituents -= {"WBA", "UTX"}
    return constituents

### Feature Engineering

3 Features
12-1
vol60
RSI

In [ ]:
FEATURE_COLS = ["mom_12_1", "vol_60", "rsi"]

In [ ]:
def compute_rsi(prices_arr: np.ndarray, period: int) -> float:
    """RSI computed from the last (period+1) closing prices."""
    delta  = np.diff(prices_arr[-(period + 1):])
    gains  = delta[delta > 0].sum() / period
    losses = -delta[delta < 0].sum() / period
    if losses == 0:
        return 100.0
    return 100.0 - 100.0 / (1.0 + gains / losses)

In [ ]:
def compute_features(
    prices: pd.DataFrame,
    lookback: int   = 252,
    vol_window: int = 60,
    rsi_period: int = 14,
    skip: int       = 21,
    fwd_days: int   = 5,
) -> pd.DataFrame:
    log_rets = np.log(prices / prices.shift(1))
    min_day  = lookback + skip + rsi_period + 1
    records  = []

    for di in range(min_day, len(prices) - fwd_days):
        date     = prices.index[di]
        fwd_di   = di + fwd_days

        # ── Point-in-time universe: only stocks IN the index on this date ──
        pit_tickers = get_constituents_on_date(date)
        # Further filter to tickers actually present in the price DataFrame
        pit_tickers = [t for t in pit_tickers if t in prices.columns]

        row_data = []
        for t in pit_tickers:
            px = prices[t].values

            # Skip if price history is too short for this ticker
            if di - lookback - skip < 0 or np.isnan(px[di]):
                continue

            mom = px[di - skip] / px[di - lookback - skip] - 1.0

            r_slice = log_rets[t].iloc[di - vol_window: di].values
            if np.any(np.isnan(r_slice)):
                continue
            vol = float(np.std(r_slice)) * np.sqrt(252.0)

            rsi = compute_rsi(px[: di + 1], rsi_period)
            fwd = px[fwd_di] / px[di] - 1.0

            row_data.append({
                "date": date, "ticker": t,
                "mom_12_1": mom, "vol_60": vol, "rsi": rsi,
                "fwd_ret": fwd,
            })

        df_row = pd.DataFrame(row_data)
        if df_row.empty:
            continue
        df_row["fwd_rank"] = df_row["fwd_ret"].rank(pct=True, na_option="keep")
        df_row = df_row.dropna(subset=["fwd_ret", "fwd_rank"] + FEATURE_COLS)
        if len(df_row) >= 5:
            records.append(df_row)

    return pd.concat(records, ignore_index=True) if records else pd.DataFrame()

### Regime Detection
Primary issue with momentum based trading is when the momentum stops...

Unless you've got a backstop in to change course on a crash, the trading strategy will do very poorly.

In [ ]:
def classify_regimes(
    prices: pd.DataFrame,
    vol_window: int  = 20,    # rolling window for market vol estimate
    dd_window: int   = 60,    # rolling window for drawdown
    vol_crash: float = 0.28,  # annualised vol threshold → crash
    vol_vol:   float = 0.17,  # annualised vol threshold → volatile
    dd_crash:  float = -0.13, # drawdown threshold → crash
    dd_vol:    float = -0.07, # drawdown threshold → volatile
) -> pd.Series:
    """
    Classify each trading day as 'trending', 'volatile', or 'crash'.

    State logic (evaluated in order, crash takes priority):
        crash    : rolling vol > vol_crash  OR  rolling DD < dd_crash
        volatile : rolling vol > vol_vol    OR  rolling DD < dd_vol
        trending : otherwise

    Uses an equal-weight index of all 30 stocks as the market proxy.
    """
    # Build a daily equal-weight index using only PIT constituents
    pit_index = []
    for date in prices.index:
        members = get_constituents_on_date(date)
        valid = [t for t in members if t in prices.columns and pd.notna(prices.loc[date, t])]
        pit_index.append(prices.loc[date, valid].mean() if valid else np.nan)
    mkt = pd.Series(pit_index, index=prices.index).ffill()
    log_rets = np.log(mkt / mkt.shift(1))
    roll_vol = log_rets.rolling(vol_window).std() * np.sqrt(252)
    roll_pk  = mkt.rolling(dd_window).max()
    roll_dd  = (mkt - roll_pk) / roll_pk

    regimes = pd.Series("trending", index=prices.index, dtype=str)
    regimes[(roll_vol > vol_vol)   | (roll_dd < dd_vol)]   = "volatile"
    regimes[(roll_vol > vol_crash) | (roll_dd < dd_crash)] = "crash"    # overrides volatile
    return regimes

### Building your Portfolio

- Use XGBoost to rank the stocks, ridge regression to optimize the portfolio weights

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 4.  XGBOOST CROSS-SECTIONAL RANKER
# ══════════════════════════════════════════════════════════════════════════════

def train_xgboost(train_df: pd.DataFrame) -> xgb.XGBRegressor:
    """
    Train XGBoost to predict the cross-sectional forward-return rank.

    Target  : fwd_rank  (percentile, 0–1)  — treated as a regression target.
              Higher rank = stronger outperformer.
    Features: mom_12_1, vol_60, rsi

    XGBoost is used in regression mode (reg:squarederror).
    For a production system you'd use xgb.train() with 'rank:pairwise'
    (LambdaMART), but that requires query groups.  Regression on rank
    percentile is the standard simplified approach.
    """
    clean = train_df[FEATURE_COLS + ["fwd_rank"]].replace([np.inf, -np.inf], np.nan).dropna()
    X = clean[FEATURE_COLS].values.astype(np.float32)
    y = clean["fwd_rank"].values.astype(np.float32)
    if len(X) == 0:
      raise ValueError("Training data is empty after cleaning — check compute_features output.")

    model = xgb.XGBRegressor(
        n_estimators     = 300,
        max_depth        = 3,          # shallow trees → low variance
        learning_rate    = 0.05,
        subsample        = 0.8,        # row subsampling per tree
        colsample_bytree = 1.0,        # use all 3 features (small set)
        min_child_weight = 5,
        reg_lambda       = 1.0,        # L2 regularisation on leaf weights
        reg_alpha        = 0.1,        # L1 regularisation
        objective        = "reg:squarederror",
        eval_metric      = "rmse",
        random_state     = 42,
        n_jobs           = -1,
        verbosity        = 0,
    )
    model.fit(X, y)
    return model


def predict_scores(model: xgb.XGBRegressor, feat_df: pd.DataFrame) -> np.ndarray:
    """Return XGBoost predicted rank scores for a snapshot of stocks."""
    X = feat_df[FEATURE_COLS].values.astype(np.float32)
    return model.predict(X)

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# 5.  RIDGE PORTFOLIO OPTIMISATION
# ══════════════════════════════════════════════════════════════════════════════

def ridge_optimize(
    scores: np.ndarray,
    tickers: list,
    ridge_lambda: float = 0.10,
    top_n: int          = 7,
) -> dict:
    """
    Ridge-regularised long-only portfolio.

    Objective:  max  Σ score_i * w_i  -  λ * ||w||²
    Subject to: Σ w_i = 1,  w_i ≥ 0

    Closed-form solution for selected subset:
        w_i* ∝ max(0, score_i) / (2λ)
    then project to the probability simplex (normalise to sum = 1).

    Higher λ  → weights shrink toward equal-weight across top-N
    Lower  λ  → weights concentrate on highest-scoring stocks
    """
    ranked_idx        = np.argsort(scores)[::-1][:top_n]
    selected_tickers  = [tickers[i] for i in ranked_idx]
    selected_scores   = scores[ranked_idx]

    raw_w = np.maximum(0.0, selected_scores) / (2.0 * ridge_lambda)
    total = raw_w.sum()
    w     = raw_w / total if total > 0 else np.full(top_n, 1.0 / top_n)

    return dict(zip(selected_tickers, w))



### BackTest the Thing


In [ ]:
def run_backtest(
    prices: pd.DataFrame,
    features_df: pd.DataFrame,
    regimes: pd.Series,
    rsi_period:   int   = 14,
    rebal_freq:   int   = 4,
    ridge_lambda: float = 0.10,
    top_n:        int   = 7,
    train_frac:   float = 0.40,
    val_frac:     float = 0.30, # New parameter
    mode:         str   = "val", # New parameter
) -> dict:
    rebal_days   = rebal_freq * 5
    unique_dates = np.sort(features_df["date"].unique())
    n            = len(unique_dates)
    # In run_backtest(), replace the percentage-based split with:
    train_cutoff = pd.Timestamp("2020-01-01")
    val_cutoff   = pd.Timestamp("2022-01-01")

    # Train: 2016–2020 (includes 2018 selloff)
    # Val:   2020–2022 (includes COVID crash and recovery)
    # Test:  2022–2026 (includes 2022 bear market AND the bull run)

    train_dates = unique_dates[unique_dates <  train_cutoff]
    val_dates   = unique_dates[(unique_dates >= train_cutoff) & (unique_dates < val_cutoff)]
    test_dates  = unique_dates[unique_dates >= val_cutoff]

    eval_dates  = val_dates if mode == "val" else test_dates # Updated logic

    if len(train_dates) < 20 or len(eval_dates) < 20: # Changed test_dates to eval_dates
        return {"sharpe": -99.0, "cagr": -99.0, "max_dd": -99.0}

    train_df = features_df[features_df["date"].isin(train_dates)]
    model    = train_xgboost(train_df)
    test_df  = features_df[features_df["date"].isin(eval_dates)] # Changed test_dates to eval_dates

    test_start  = pd.Timestamp(eval_dates[0]) # Changed test_dates to eval_dates
    test_prices = prices[prices.index >= test_start].copy()

    weights    = {}   # start empty — filled on first rebalance
    strat_val  = 100.0
    bench_val  = 100.0
    peak       = 100.0
    max_dd     = 0.0
    kills      = 0
    last_rebal = -rebal_days

    strat_curve, bench_curve, date_index, regime_log = [], [], [], []

    for di in range(1, len(test_prices)):
        date = test_prices.index[di]

        # ── Point-in-time universe for today ──────────────────────────────
        pit_tickers = get_constituents_on_date(date)
        pit_tickers = [t for t in pit_tickers if t in test_prices.columns]

        daily_rets = {}
        for t in pit_tickers:
            p0 = test_prices[t].iloc[di - 1]
            p1 = test_prices[t].iloc[di]
            if pd.notna(p0) and pd.notna(p1) and p0 > 0:
                daily_rets[t] = p1 / p0 - 1.0

        reg = regimes.get(date, "trending")
        regime_log.append(reg)

        if di - last_rebal >= rebal_days:
            last_rebal = di
            avail = test_df[test_df["date"] <= date]
            if not avail.empty:
                snap_date = avail["date"].max()
                # Only score stocks in today's PIT universe
                snap = (
                    test_df[
                        (test_df["date"] == snap_date) &
                        (test_df["ticker"].isin(pit_tickers))
                    ]
                    .set_index("ticker")
                    .dropna(subset=FEATURE_COLS)
                )

                if len(snap) >= top_n:
                    scores_arr    = predict_scores(model, snap.reset_index())
                    tickers_avail = snap.index.tolist()

                    if reg == "crash":
                        weights = {}
                        kills  += 1
                    else:
                        scale = 0.70 if reg == "volatile" else 1.00
                        opt_w = ridge_optimize(scores_arr, tickers_avail,
                                               ridge_lambda, top_n)
                        weights = {t: w * scale for t, w in opt_w.items()}

        # P&L — only on held positions that have valid prices today
        strat_ret = sum(
            weights.get(t, 0.0) * daily_rets.get(t, 0.0)
            for t in weights
        )
        # Benchmark: equal-weight PIT universe
        bench_ret = float(np.mean([
            daily_rets[t] for t in pit_tickers if t in daily_rets
        ]))
        strat_val *= (1.0 + strat_ret)
        bench_val *= (1.0 + bench_ret)
        peak       = max(peak, strat_val)
        max_dd     = min(max_dd, (strat_val - peak) / peak)

        strat_curve.append(strat_val)
        bench_curve.append(bench_val)
        date_index.append(date)


    if len(strat_curve) < 50:
        return {"sharpe": -99.0, "cagr": -99.0, "max_dd": -99.0}

    # ── Performance metrics ───────────────────────────────────────────────────
    daily_rets_arr = np.diff(strat_curve) / np.array(strat_curve[:-1])
    n_years        = len(strat_curve) / 252.0
    cagr           = (strat_val / 100.0) ** (1.0 / n_years) - 1.0
    bench_cagr     = (bench_val / 100.0) ** (1.0 / n_years) - 1.0
    sharpe         = (
        (daily_rets_arr.mean() / daily_rets_arr.std()) * np.sqrt(252.0)
        if daily_rets_arr.std() > 0 else 0.0
    )

    # Feature importance from XGBoost (gain-based)
    feat_imp = dict(zip(FEATURE_COLS, model.feature_importances_))

    return {
        "sharpe":      round(float(sharpe),     4),
        "cagr":        round(float(cagr * 100), 2),
        "bench_cagr":  round(float(bench_cagr * 100), 2),
        "max_dd":      round(float(max_dd * 100), 2),
        "cum_ret":     round(strat_val - 100.0, 2),
        "kills":       kills,
        "strat_curve": strat_curve,
        "bench_curve": bench_curve,
        "date_index":  date_index,
        "regime_log":  regime_log,
        "feat_imp":    feat_imp,
        "model":       model,
        "weights":     weights,
    }

### Param Optimization

Parameters were fine-tuning:
1. RSI window
2. Ridge regression lambda
3. Number of stocks to go long

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# 7.  OPTUNA HYPERPARAMETER OPTIMISATION
# ══════════════════════════════════════════════════════════════════════════════
#
# Search space:
#   rsi_period   : [7, 10, 14, 21, 28]
#   rebal_freq   : [1, 2, 4, 6, 8, 12]  weeks
#   ridge_lambda : log-uniform [0.01, 1.0]
#   top_n        : [3, 5, 7, 9, 12, 15]
#
# Objective      : maximise Sharpe ratio (walk-forward backtest)
# Sampler        : TPE (Tree-structured Parzen Estimator) — Bayesian, not grid
# Pruner         : MedianPruner — stop unpromising trials early
#
# Note: each trial recomputes features for the sampled rsi_period,
#       so we cache feature DataFrames by rsi_period to save time.
# ─────────────────────────────────────────────────────────────────────────────

N_OPTUNA_TRIALS = 5   # increase to 150-200 for more thorough search

def run_optuna(
    prices: pd.DataFrame,
    regimes: pd.Series,
    n_trials: int = N_OPTUNA_TRIALS,
    seed: int = 42,
) -> tuple[optuna.Study, dict]:
    """
    Run Optuna TPE search. Returns (study, best_params).
    """
    # Cache feature DataFrames per RSI period (avoids recomputing on every trial)
    feat_cache: dict[int, pd.DataFrame] = {}

    def objective(trial: optuna.Trial) -> float:
        rsi_period   = trial.suggest_categorical("rsi_period",   [7, 10, 14, 21, 28])
        rebal_freq   = trial.suggest_categorical("rebal_freq",   [1, 2, 4, 6, 8, 12])
        ridge_lambda = trial.suggest_float(      "ridge_lambda", 0.01, 1.0, log=True)
        top_n        = trial.suggest_categorical("top_n",        [3, 5, 7, 9, 12, 15])

        if rsi_period not in feat_cache:
            feat_cache[rsi_period] = compute_features(
                prices, lookback=252, vol_window=60, rsi_period=rsi_period
            )

        result = run_backtest(
            prices,
            feat_cache[rsi_period],
            regimes,
            rsi_period   = rsi_period,
            rebal_freq   = rebal_freq,
            ridge_lambda = ridge_lambda,
            top_n        = top_n,
            mode         = "val", # Added mode="val"
        )
        sharpe = result["sharpe"]
        # Report for pruning (Optuna can cut bad trials early)
        trial.report(sharpe, step=0)
        return sharpe if np.isfinite(sharpe) else -99.0

    sampler = TPESampler(seed=seed)
    pruner  = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=0)
    study   = optuna.create_study(
        direction = "maximize",
        sampler   = sampler,
        pruner    = pruner,
        study_name= "momentum_dow30",
    )

    print(f"\n{'='*60}")
    print(f"  Optuna TPE search — {n_trials} trials")
    print(f"  RSI period   : [7, 10, 14, 21, 28]")
    print(f"  Rebal freq   : [1, 2, 4, 6, 8, 12] weeks")
    print(f"  Ridge lambda : log-uniform [0.01, 1.0]")
    print(f"  Top-N longs  : [3, 5, 7, 9, 12, 15]")
    print(f"{'='*60}\n")

    def progress_cb(study, trial):
        if trial.number % 10 == 0 or trial.number == 0:
            best = study.best_value if study.best_trial else float("nan")
            v    = trial.value if trial.value is not None else float("nan")
            p    = trial.params
            print(
                f"  Trial {trial.number+1:3d}/{n_trials}"
                f"  RSI={p.get('rsi_period','?'):2}  "
                f"rebal={p.get('rebal_freq','?')}w  "
                f"λ={p.get('ridge_lambda',0):.3f}  "
                f"N={p.get('top_n','?'):2}  "
                f"Sharpe={v:.3f}  (best={best:.3f})"
            )

    study.optimize(objective, n_trials=n_trials, callbacks=[progress_cb])

    best_params = study.best_params
    print(f"\n{'='*60}")
    print(f"  BEST TRIAL  #{study.best_trial.number + 1}")
    print(f"{'='*60}")
    for k, v in best_params.items():
        print(f"    {k:20s}: {v}")
    print(f"    {'Sharpe':20s}: {study.best_value:.4f}")

    return study, best_params


# ══════════════════════════════════════════════════════════════════════════════
# 8.  VISUALISATION
# ══════════════════════════════════════════════════════════════════════════════

DARK    = "#0a0c0f"
SURFACE = "#111418"
BORDER  = "#232830"
TEXT    = "#e2e8f0"
MUTED   = "#8896a8"
GREEN   = "#22c55e"
RED     = "#ef4444"
AMBER   = "#f59e0b"
BLUE    = "#60a5fa"
PURPLE  = "#a78bfa"

plt.rcParams.update({
    "figure.facecolor": DARK,    "axes.facecolor":  SURFACE,
    "axes.edgecolor":   BORDER,  "axes.labelcolor": MUTED,
    "xtick.color":      MUTED,   "ytick.color":     MUTED,
    "text.color":       TEXT,    "grid.color":      BORDER,
    "grid.linewidth":   0.5,     "font.family":     "monospace",
    "axes.titlecolor":  TEXT,    "axes.titlesize":  10,
    "axes.titleweight": "bold",
})


def plot_backtest(result: dict, best_params: dict,
                  save_path: str = "results/backtest_summary.png"):
    """6-panel backtest dashboard."""
    fig = plt.figure(figsize=(16, 12), facecolor=DARK)
    gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35,
                            left=0.07, right=0.97, top=0.92, bottom=0.07)

    dates   = pd.DatetimeIndex(result["date_index"])
    strat   = np.array(result["strat_curve"])
    bench   = np.array(result["bench_curve"])
    regimes = result["regime_log"]

    reg_colors = {"trending": GREEN, "volatile": AMBER, "crash": RED}

    # ── Panel 1: Cumulative performance (full width) ──────────────────────────
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(dates, strat, color=GREEN, lw=2.0,
             label=f"Strategy   CAGR {result['cagr']:.1f}%", zorder=3)
    ax1.plot(dates, bench, color=BLUE,  lw=1.4, ls="--", alpha=0.7,
             label=f"Benchmark  CAGR {result['bench_cagr']:.1f}%", zorder=2)

    # Shade regime periods
    prev_reg, seg_start = regimes[0], dates[0]
    for i in range(1, len(dates)):
        if regimes[i] != prev_reg or i == len(dates) - 1:
            ax1.axvspan(seg_start, dates[i], alpha=0.09,
                        color=reg_colors[prev_reg], zorder=1)
            seg_start = dates[i]
            prev_reg  = regimes[i]

    legend_lines  = ax1.get_lines()
    legend_patches = [
        Patch(facecolor=GREEN, alpha=0.4, label="Trending"),
        Patch(facecolor=AMBER, alpha=0.4, label="Volatile (70% size)"),
        Patch(facecolor=RED,   alpha=0.4, label="Crash → cash"),
    ]
    ax1.legend(handles=[*legend_lines, *legend_patches],
               loc="upper left", fontsize=8.5,
               facecolor=SURFACE, edgecolor=BORDER, labelcolor=TEXT, ncol=2)
    ax1.set_title("CUMULATIVE PERFORMANCE  (shaded = regime)", pad=8)
    ax1.set_ylabel("Portfolio value  (base = 100)")
    ax1.grid(True, alpha=0.3)

    # ── Panel 2: Drawdown ─────────────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[1, :2])
    peak_arr = np.maximum.accumulate(strat)
    dd_arr   = (strat - peak_arr) / peak_arr * 100.0
    ax2.fill_between(dates, dd_arr, 0, color=RED, alpha=0.45)
    ax2.plot(dates, dd_arr, color=RED, lw=0.8)
    ax2.set_title(f"DRAWDOWN  (max {result['max_dd']:.1f}%)")
    ax2.set_ylabel("%")
    ax2.grid(True, alpha=0.3)

    # ── Panel 3: XGBoost feature importance ───────────────────────────────────
    ax3 = fig.add_subplot(gs[1, 2])
    feat_imp = result["feat_imp"]
    labels   = [f"12-1 Mom", "Vol 60d", f"RSI {best_params['rsi_period']}d"]
    vals     = [feat_imp.get(f, 0.0) for f in FEATURE_COLS]
    vals_pct = np.array(vals) / sum(vals) * 100.0
    colors_fi = [GREEN, BLUE, PURPLE]
    bars = ax3.barh(labels, vals_pct, color=colors_fi, height=0.5)
    for bar, v in zip(bars, vals_pct):
        ax3.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                 f"{v:.1f}%", va="center", fontsize=9, color=TEXT)
    ax3.set_title("XGBOOST FEATURE IMPORTANCE  (gain %)")
    ax3.set_xlim(0, max(vals_pct) * 1.3)
    ax3.grid(True, alpha=0.3, axis="x")

    # ── Panel 4: Regime timeline ──────────────────────────────────────────────
    ax4 = fig.add_subplot(gs[2, :2])
    reg_int = [{"trending": 1, "volatile": 2, "crash": 3}[r] for r in regimes]
    cmap_r  = {1: GREEN, 2: AMBER, 3: RED}
    for i in range(len(dates) - 1):
        ax4.axvspan(dates[i], dates[i + 1], ymin=0, ymax=1,
                    color=cmap_r[reg_int[i]], alpha=0.75)
    ax4.set_yticks([])
    ax4.set_title(f"REGIME TIMELINE  (kill-switch fired {result['kills']}×)")
    n = len(regimes)
    ax4.legend(handles=[
        Patch(color=GREEN, label=f"Trending  {regimes.count('trending')/n*100:.0f}%"),
        Patch(color=AMBER, label=f"Volatile  {regimes.count('volatile')/n*100:.0f}%"),
        Patch(color=RED,   label=f"Crash     {regimes.count('crash')/n*100:.0f}%"),
    ], loc="upper right", fontsize=8, facecolor=SURFACE,
       edgecolor=BORDER, labelcolor=TEXT)

    # ── Panel 5: Metrics + best params ───────────────────────────────────────
    ax5 = fig.add_subplot(gs[2, 2])
    ax5.axis("off")
    rows = [
        ("Ann. return",  f"{result['cagr']:.2f}%",       GREEN),
        ("Benchmark",    f"{result['bench_cagr']:.2f}%",  BLUE),
        ("Sharpe ratio", f"{result['sharpe']:.4f}",
         GREEN if result["sharpe"] > 0.5 else AMBER),
        ("Max drawdown", f"{result['max_dd']:.2f}%",      RED),
        ("Cumul. return",f"{result['cum_ret']:.1f}%",
         GREEN if result["cum_ret"] > 0 else RED),
        ("Kill-switches",f"{result['kills']}",            PURPLE),
        ("── best params ──", "", MUTED),
        ("RSI period",   f"{best_params['rsi_period']}d", TEXT),
        ("Rebal freq",   f"{best_params['rebal_freq']}w", TEXT),
        ("Ridge λ",      f"{best_params['ridge_lambda']:.4f}", TEXT),
        ("Top-N longs",  f"{best_params['top_n']}",       TEXT),
    ]
    for i, (label, val, color) in enumerate(rows):
        y = 1.0 - i * 0.092
        ax5.text(0.02, y, label, transform=ax5.transAxes,
                 fontsize=9, color=MUTED, va="top")
        ax5.text(0.98, y, val,   transform=ax5.transAxes,
                 fontsize=9, color=color, va="top", ha="right", fontweight="bold")

    fig.suptitle(
        "DOW 30 MOMENTUM  //  XGBoost Ranking  +  Ridge Optimisation  +  HMM Regime Detection",
        fontsize=11.5, fontweight="bold", color=TEXT, y=0.97
    )
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=DARK)
    print(f"\n  → Saved: {save_path}")
    plt.close()



def plot_optuna(study: optuna.Study,
                save_path: str = "results/optuna_analysis.png"):
    """
    4-panel Optuna analysis:
      1. Optimisation history (Sharpe per trial)
      2. Parallel coordinates coloured by Sharpe
      3. Marginal importance of each hyperparameter
      4. Top-20 trials bar chart
    """
    trials_df = study.trials_dataframe(attrs=("number","value","params","state"))
    trials_df = trials_df[trials_df["state"] == "COMPLETE"].copy()
    trials_df.rename(columns={"value": "sharpe"}, inplace=True)
    trials_df.sort_values("sharpe", ascending=False, inplace=True)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10), facecolor=DARK)
    fig.suptitle("OPTUNA HYPERPARAMETER SEARCH  //  objective: Sharpe ratio",
                 fontsize=11, fontweight="bold", color=TEXT, y=0.98)

    cmap = LinearSegmentedColormap.from_list("rg", [RED, AMBER, GREEN])
    vmin = trials_df["sharpe"].quantile(0.10)
    vmax = trials_df["sharpe"].quantile(0.90)

    # ── 1. Optimisation history ────────────────────────────────────────────────
    ax = axes[0, 0]
    ax.scatter(trials_df["number"] + 1, trials_df["sharpe"],
               c=trials_df["sharpe"], cmap=cmap, vmin=vmin, vmax=vmax,
               s=25, alpha=0.8, zorder=3)
    # running best
    running_best = trials_df.set_index("number")["sharpe"].sort_index().cummax()
    ax.plot(running_best.index + 1, running_best.values,
            color=GREEN, lw=1.5, ls="--", zorder=4, label="Running best")
    ax.axhline(0.5, color=AMBER, lw=0.8, ls=":", alpha=0.7)
    ax.set_xlabel("Trial number", color=MUTED)
    ax.set_ylabel("Sharpe ratio", color=MUTED)
    ax.set_title("OPTIMISATION HISTORY", color=TEXT)
    ax.legend(fontsize=8, facecolor=SURFACE, edgecolor=BORDER, labelcolor=TEXT)
    ax.grid(True, alpha=0.3)

    # ── 2. RSI period × Rebal freq heat-map ───────────────────────────────────
    ax = axes[0, 1]
    p_rsi   = "params_rsi_period"
    p_rebal = "params_rebal_freq"
    p_n     = "params_top_n"
    p_lam   = "params_ridge_lambda"

    if p_rsi in trials_df.columns and p_rebal in trials_df.columns:
        pivot = trials_df.groupby([p_rsi, p_rebal])["sharpe"].mean().unstack(fill_value=np.nan)
        im = ax.imshow(pivot.values, cmap=cmap, aspect="auto", vmin=vmin, vmax=vmax)
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels([f"{c}w" for c in pivot.columns], fontsize=9)
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index, fontsize=9)
        ax.set_xlabel("Rebalance frequency", color=MUTED)
        ax.set_ylabel("RSI period", color=MUTED)
        ax.set_title("RSI × REBAL FREQ  (mean Sharpe)", color=TEXT)
        for i in range(len(pivot.index)):
            for j in range(len(pivot.columns)):
                v = pivot.values[i, j]
                if not np.isnan(v):
                    ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                            fontsize=7.5,
                            color="black" if v > trials_df["sharpe"].median() else "white")
        plt.colorbar(im, ax=ax, shrink=0.85).ax.tick_params(labelcolor=MUTED)

    # ── 3. Scatter: CAGR vs Sharpe, sized by top_n ───────────────────────────
    ax = axes[1, 0]
    p_cagr = "user_attrs_cagr"   # not stored by default; use sharpe as proxy
    x_vals  = trials_df["sharpe"]
    y_vals  = trials_df["sharpe"].rank()
    size    = (trials_df[p_n] if p_n in trials_df.columns else pd.Series(30, index=trials_df.index))
    sc = ax.scatter(trials_df["number"] + 1, x_vals,
                    c=trials_df[p_lam] if p_lam in trials_df.columns else x_vals,
                    cmap=cmap, s=30, alpha=0.7, edgecolors="none")
    plt.colorbar(sc, ax=ax, label="Ridge λ").ax.tick_params(labelcolor=MUTED)
    ax.set_xlabel("Trial number", color=MUTED)
    ax.set_ylabel("Sharpe ratio", color=MUTED)
    ax.set_title("SHARPE BY TRIAL  (colour = Ridge λ)", color=TEXT)
    ax.axhline(0.5, color=GREEN, ls="--", lw=0.8, alpha=0.5)
    ax.grid(True, alpha=0.3)

    # ── 4. Top-20 trials bar chart ────────────────────────────────────────────
    ax = axes[1, 1]
    top20 = trials_df.head(20).copy()
    bar_colors = [GREEN if s > 0.5 else AMBER if s > 0.3 else RED
                  for s in top20["sharpe"]]
    ax.bar(range(len(top20)), top20["sharpe"], color=bar_colors, width=0.7)
    labels_20 = []
    for _, r in top20.iterrows():
        rsi = int(r.get(p_rsi, 0))
        rb  = int(r.get(p_rebal, 0))
        lam = r.get(p_lam, 0)
        n   = int(r.get(p_n, 0))
        labels_20.append(f"R{rsi} {rb}w\nλ{lam:.2f} N{n}")
    ax.set_xticks(range(len(top20)))
    ax.set_xticklabels(labels_20, rotation=90, fontsize=6.5)
    ax.set_ylabel("Sharpe ratio", color=MUTED)
    ax.set_title("TOP 20 TRIALS BY SHARPE", color=TEXT)
    ax.axhline(0.5, color=GREEN, ls="--", lw=0.8, alpha=0.5)
    ax.grid(True, alpha=0.3, axis="y")

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=DARK)
    print(f"  → Saved: {save_path}")
    plt.close()

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# 9.  MAIN
# ══════════════════════════════════════════════════════════════════════════════

def main():
    print("\n" + "="*60)
    print("  DOW 30 MOMENTUM STRATEGY  —  FULL PIPELINE")
    print("="*60)

   # ── Step 1: Fetch prices from yfinance ────────────────────────────────────
    print("\n[1/5]  Downloading 10Y of daily prices from Yahoo Finance ...")
    prices = fetch_prices()

    print(f"       {len(prices)} trading days  ×  {len(prices.columns)} tickers in price history")
    print(f"       {prices.index[0].date()}  →  {prices.index[-1].date()}")

    # ── Step 2: Regime detection ──────────────────────────────────────────────
    print("\n[2/5]  Detecting regimes ...")
    regimes = classify_regimes(prices)
    for state in ["trending", "volatile", "crash"]:
        n = (regimes == state).sum()
        print(f"       {state:10s}: {n:4d} days  ({n/len(regimes)*100:.1f}%)")
    # Added diagnostic print statements
    crash_days = regimes[regimes == "crash"]
    volatile_days = regimes[regimes == "volatile"]
    print(f"       Crash days: {len(crash_days)} — first: {crash_days.index.min() if len(crash_days) > 0 else 'none'}, last: {crash_days.index.max() if len(crash_days) > 0 else 'none'}")
    print(f"       Volatile days: {len(volatile_days)}")
    print("       Crash periods should cover approximately: Dec 2018, Feb-Mar 2020, Jan-Oct 2022")


    # ── Step 3: Optuna hyperparameter search ──────────────────────────────────
    print("\n[3/5]  Running Optuna TPE hyperparameter search ...")
    study, best_params = run_optuna(prices, regimes, n_trials=N_OPTUNA_TRIALS)

    # Save Optuna results table
    trials_df = study.trials_dataframe()
    trials_df.to_csv("results/optuna_trials.csv", index=False)
    print(f"\n       All trial results  → results/optuna_trials.csv")

    # ── Step 4: Full backtest with best params ────────────────────────────────
    # Changed print statement and added mode="test"
    print("\n[4/5]  Final out-of-sample test (held-out 30% never seen by Optuna) ...")
    feat_df = compute_features(
        prices,
        lookback    = 252,
        vol_window  = 60,
        rsi_period  = best_params["rsi_period"],
    )
    result = run_backtest(
        prices, feat_df, regimes,
        rsi_period   = best_params["rsi_period"],
        rebal_freq   = best_params["rebal_freq"],
        ridge_lambda = best_params["ridge_lambda"],
        top_n        = best_params["top_n"],
        mode         = "test", # Added mode="test"
    )

    print(f"\n       ── Final metrics ──────────────────────────────")
    print(f"       Ann. return  : {result['cagr']:.2f}%"
          f"  (benchmark {result['bench_cagr']:.2f}%)")
    print(f"       Sharpe ratio : {result['sharpe']:.4f}")
    print(f"       Max drawdown : {result['max_dd']:.2f}%")
    print(f"       Cumulative   : {result['cum_ret']:.1f}%")
    print(f"       Kill-switches: {result['kills']}×")

    # ── Step 5: Charts ────────────────────────────────────────────────────────
    print("\n[5/5]  Generating charts ...")
    # add one line directly above it:
    globals()["result"] = result   # ← expose to notebook scope
    plot_backtest(result, best_params)


    plot_optuna(study)

    # Save best params text file
    with open("results/best_params.txt", "w") as f:
        f.write("BEST HYPERPARAMETERS  (Optuna TPE)\n")
        f.write("=" * 40 + "\n")
        for k, v in best_params.items():
            f.write(f"{k:20s}: {v}\n")
        f.write("\nPERFORMANCE METRICS\n")
        f.write("=" * 40 + "\n")
        f.write(f"{'sharpe':20s}: {result['sharpe']:.4f}\n")
        f.write(f"{'cagr_%':20s}: {result['cagr']:.2f}\n")
        f.write(f"{'bench_cagr_%':20s}: {result['bench_cagr']:.2f}\n")
        f.write(f"{'max_drawdown_%':20s}: {result['max_dd']:.2f}\n")
        f.write(f"{'cumulative_%':20s}: {result['cum_ret']:.2f}\n")
        f.write(f"{'kill_switches':20s}: {result['kills']}\n")
    print("       → Saved: results/best_params.txt")

    print(f"\n{'='*60}")
    print("  Done. All outputs written to  results/")
    print("="*60 + "\n")
    return result, study, prices, regimes, best_params

if __name__ == "__main__":
    result, study, prices, regimes, best_params = main()


  DOW 30 MOMENTUM STRATEGY  —  FULL PIPELINE

[1/5]  Downloading 10Y of daily prices from Yahoo Finance ...
       2526 trading days  ×  37 tickers in price history
       2016-04-01  →  2026-04-17

[2/5]  Detecting regimes ...


[I 2026-04-21 01:09:22,816] A new study created in memory with name: momentum_dow30


       trending  : 1895 days  (75.0%)
       volatile  :  451 days  (17.9%)
       crash     :  180 days  (7.1%)
       Crash days: 180 — first: 2018-02-09 00:00:00, last: 2025-05-07 00:00:00
       Volatile days: 451
       Crash periods should cover approximately: Dec 2018, Feb-Mar 2020, Jan-Oct 2022

[3/5]  Running Optuna TPE hyperparameter search ...

  Optuna TPE search — 5 trials
  RSI period   : [7, 10, 14, 21, 28]
  Rebal freq   : [1, 2, 4, 6, 8, 12] weeks
  Ridge lambda : log-uniform [0.01, 1.0]
  Top-N longs  : [3, 5, 7, 9, 12, 15]



[I 2026-04-21 01:09:42,697] Trial 0 finished with value: 0.4543 and parameters: {'rsi_period': 10, 'rebal_freq': 4, 'ridge_lambda': 0.8706020878304853, 'top_n': 3}. Best is trial 0 with value: 0.4543.


  Trial   1/5  RSI=10  rebal=4w  λ=0.871  N= 3  Sharpe=0.454  (best=0.454)


[I 2026-04-21 01:10:03,317] Trial 1 finished with value: 0.5435 and parameters: {'rsi_period': 14, 'rebal_freq': 4, 'ridge_lambda': 0.012385137298860933, 'top_n': 12}. Best is trial 1 with value: 0.5435.
[I 2026-04-21 01:10:05,261] Trial 2 finished with value: 0.5284 and parameters: {'rsi_period': 14, 'rebal_freq': 4, 'ridge_lambda': 0.10968217207529521, 'top_n': 7}. Best is trial 1 with value: 0.5435.
[I 2026-04-21 01:10:07,363] Trial 3 finished with value: 0.29 and parameters: {'rsi_period': 10, 'rebal_freq': 6, 'ridge_lambda': 0.1217284708112243, 'top_n': 9}. Best is trial 1 with value: 0.5435.
[I 2026-04-21 01:10:10,459] Trial 4 finished with value: 0.2382 and parameters: {'rsi_period': 10, 'rebal_freq': 6, 'ridge_lambda': 0.0134003672433548, 'top_n': 12}. Best is trial 1 with value: 0.5435.



  BEST TRIAL  #2
    rsi_period          : 14
    rebal_freq          : 4
    ridge_lambda        : 0.012385137298860933
    top_n               : 12
    Sharpe              : 0.5435

       All trial results  → results/optuna_trials.csv

[4/5]  Final out-of-sample test (held-out 30% never seen by Optuna) ...

       ── Final metrics ──────────────────────────────
       Ann. return  : 8.25%  (benchmark 10.94%)
       Sharpe ratio : 0.6359
       Max drawdown : -21.79%
       Cumulative   : 40.2%
       Kill-switches: 1×

[5/5]  Generating charts ...

  → Saved: results/backtest_summary.png
  → Saved: results/optuna_analysis.png
       → Saved: results/best_params.txt

  Done. All outputs written to  results/



In [ ]:
# ── Diagnostics — run this cell after main() completes ──────────────────────

import numpy as np

strat = np.array(result["strat_curve"])
bench = np.array(result["bench_curve"])
dates = pd.DatetimeIndex(result["date_index"])

rets_s = np.diff(strat) / strat[:-1]
rets_b = np.diff(bench) / bench[:-1]

print(f"Strategy  — ann. vol: {rets_s.std()*np.sqrt(252)*100:.1f}%  mean daily ret: {rets_s.mean()*252*100:.2f}%")
print(f"Benchmark — ann. vol: {rets_b.std()*np.sqrt(252)*100:.1f}%  mean daily ret: {rets_b.mean()*252*100:.2f}%")
print(f"Correlation strat vs bench: {np.corrcoef(rets_s, rets_b)[0,1]:.3f}")
print(f"Weight sum on last rebalance: {sum(result['weights'].values()):.4f}")

df_rets = pd.DataFrame({"strat": rets_s, "bench": rets_b}, index=dates[1:])
annual = df_rets.resample("YE").apply(lambda x: (1+x).prod()-1) * 100
print("\nYear-by-year returns:")
print(annual.round(1).to_string())

Strategy  — ann. vol: 14.0%  mean daily ret: 8.88%
Benchmark — ann. vol: 14.8%  mean daily ret: 11.31%
Correlation strat vs bench: 0.911
Weight sum on last rebalance: 0.7000

Year-by-year returns:
            strat  bench
2022-12-31  -12.4   -7.6
2023-12-31   14.2   18.3
2024-12-31    8.9   17.1
2025-12-31   17.9   16.7
2026-12-31    9.1    3.5


In [ ]:
# Reconstruct the benchmark manually using PIT constituents
bench_check = []
test_prices = prices[prices.index >= pd.Timestamp("2022-01-01")]

for di in range(1, len(test_prices)):
    date = test_prices.index[di]
    pit  = get_constituents_on_date(date)
    pit  = [t for t in pit if t in test_prices.columns]

    daily_r = []
    for t in pit:
        p0 = test_prices[t].iloc[di-1]
        p1 = test_prices[t].iloc[di]
        if pd.notna(p0) and pd.notna(p1) and p0 > 0:
            daily_r.append(p1/p0 - 1)

    bench_check.append(np.mean(daily_r) if daily_r else 0.0)

bench_arr = np.array(bench_check)
bench_curve = 100 * np.cumprod(1 + bench_arr)
bench_dates = pd.DatetimeIndex(test_prices.index[1:])

df_bench = pd.Series(bench_arr, index=bench_dates)
annual_bench = df_bench.resample("YE").apply(lambda x: (1+x).prod()-1) * 100
print("PIT benchmark year-by-year:")
print(annual_bench.round(1))

# Also check what the actual Dow Jones returned
# DJIA ETF as ground truth
import yfinance as yf
dia = yf.download("DIA", start="2022-01-01", end="2026-04-18",
                  auto_adjust=True, progress=False)["Close"]
dia_rets = dia.pct_change().dropna()
dia_annual = dia_rets.resample("YE").apply(lambda x: (1+x).prod()-1) * 100
print("\nActual Dow Jones (DIA ETF) year-by-year:")
print(dia_annual.round(1))

PIT benchmark year-by-year:
Date
2022-12-31    -6.9
2023-12-31    18.3
2024-12-31    17.1
2025-12-31    16.7
2026-12-31     3.5
Freq: YE-DEC, dtype: float64

Actual Dow Jones (DIA ETF) year-by-year:
Ticker       DIA
Date            
2022-12-31  -7.6
2023-12-31  16.0
2024-12-31  14.8
2025-12-31  14.7
2026-12-31   3.3


In [ ]:
print(f"Best RSI period: {best_params['rsi_period']}")
print(f"Best top-N:      {best_params['top_n']}")
print(f"Best rebal freq: {best_params['rebal_freq']}w")
print(f"Best lambda:     {best_params['ridge_lambda']:.4f}")

# Also check feature importance
print("\nFeature importance:")
for feat, imp in result['feat_imp'].items():
    print(f"  {feat}: {imp:.4f}")

Best RSI period: 14
Best top-N:      12
Best rebal freq: 4w
Best lambda:     0.0124

Feature importance:
  mom_12_1: 0.3195
  vol_60: 0.3774
  rsi: 0.3031


In [ ]:
# See the full weight distribution on the last rebalance
weights = result['weights']
nonzero = {t: w for t, w in weights.items() if w > 0.001}
sorted_w = sorted(nonzero.items(), key=lambda x: x[1], reverse=True)

print(f"Positions held: {len(nonzero)}")
print(f"Weight distribution:")
for t, w in sorted_w:
    bar = '█' * int(w * 200)
    print(f"  {t:6s}  {w*100:5.1f}%  {bar}")

print(f"\nTop 3 concentration: {sum(w for _,w in sorted_w[:3])*100:.1f}%")
print(f"Top 5 concentration: {sum(w for _,w in sorted_w[:5])*100:.1f}%")

Positions held: 12
Weight distribution:
  IBM       6.5%  █████████████
  CSCO      6.3%  ████████████
  JNJ       6.0%  ████████████
  CAT       6.0%  ████████████
  NVDA      5.9%  ███████████
  KO        5.7%  ███████████
  AXP       5.6%  ███████████
  MMM       5.6%  ███████████
  AMZN      5.6%  ███████████
  DIS       5.6%  ███████████
  SHW       5.5%  ███████████
  GS        5.5%  ███████████

Top 3 concentration: 18.8%
Top 5 concentration: 30.8%


The weight distribution is actually fine — that's a well-diversified portfolio, roughly equal-weight across 12 positions with only a small tilt toward the top-scored names. The low lambda didn't cause concentration here, it just means the weights are nearly uniform anyway because the XGBoost scores themselves are clustered close together.
But look at the holdings. IBM, CSCO, JNJ, KO, MMM, DIS — these are the classic low-volatility, defensive Dow names. The model is systematically picking the least exciting stocks in the index. That confirms the vol_60 dominance hypothesis — it's selecting low-vol defensives over high-momentum growth names. NVDA and AMZN appearing is a recent addition, likely because they joined the index and immediately had strong momentum scores that overcame the vol penalty.
Notably absent from the portfolio: MSFT, AAPL, UNH, V, HD — the Dow's best performers over 2023–2025. These were all penalised for high volatility despite being the strongest momentum names in the index.
This is your answer for why the strategy underperforms every bull market year. It's not a code bug. The model learned a low-vol factor from the training data and is using it to avoid exactly the stocks that drove benchmark returns in the test period.